# Proyecto Properatio: Creación y evaluación de model predictivo

Creacion de un modelo capaz de predecir valores en USD de propiedades tipo Oficinas y Locales Comerciales en CABA Argentina <br><br>
Creado por: Adriana Villalobos

## 1. Importación de librerías y Carga del dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import mlflow.sklearn
from mlflow.models.signature import infer_signature

In [2]:
import os
from pathlib import Path

print("Actual directorio de trabajo:", os.getcwd())

# Configuro el directorio para no tener /mlruns dentro de los notebooks
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    os.chdir(current_dir.parent)

print("Nuevo directorio de trabajo:", os.getcwd())

Actual directorio de trabajo: /Users/cosmos/Adri/Developer/DataScience/Clases y Consignas/Proyecto Final/notebooks
Nuevo directorio de trabajo: /Users/cosmos/Adri/Developer/DataScience/Clases y Consignas/Proyecto Final


In [3]:
df = pd.read_csv('data/data_cleaned.csv', sep=",")
df.head()

,neighborhood,rooms,bathrooms,surface_covered,property_type,operation_type,price_usd,price_per_m2,bathrooms_missing,rooms_missing
0,Once,2.0,1.0,20.0,Oficina,Venta,32000.000000,1600.0,0,1
1,Flores,2.0,1.0,32.0,Oficina,Alquiler,279.720280,9.0,0,1
2,Flores,2.0,1.0,46.0,Oficina,Alquiler,349.650350,8.0,0,1
3,Palermo,2.0,2.0,70.0,Oficina,Alquiler,1258.741259,18.0,0,1
4,Tribunales,2.0,1.0,40.0,Oficina,Venta,89000.000000,2225.0,0,1


In [4]:
df['operation_type'].value_counts()

operation_type
Alquiler    11600
Venta        9754
Name: count, dtype: int64

In [5]:
df['property_type'].value_counts()

property_type
Oficina            11011
Local comercial    10343
Name: count, dtype: int64

In [6]:
df.sample(5)

,neighborhood,rooms,bathrooms,surface_covered,property_type,operation_type,price_usd,price_per_m2,bathrooms_missing,rooms_missing
9614,Villa Crespo,1.0,2.0,90.0,Local comercial,Venta,170000.000000,1889.0,0,1
15176,Villa Crespo,1.0,2.0,149.0,Local comercial,Alquiler,406.504065,3.0,0,1
20353,Puerto Madero,2.0,3.0,275.0,Oficina,Alquiler,6600.000000,24.0,0,1
15444,Congreso,1.0,2.0,336.0,Local comercial,Venta,475000.000000,1414.0,0,1
11190,Palermo,1.0,2.0,180.0,Local comercial,Venta,30000.000000,167.0,0,1


In [7]:
# Incluyo mlflow para trackear el desempeño de diferentes opciones a lo largo del proceso

mlflow.set_tracking_uri(f"file://{os.getcwd()}/mlruns")
mlflow.set_experiment(experiment_name="Proyectio_Properatio")

<Experiment: artifact_location='file:///Users/cosmos/Adri/Developer/DataScience/Clases%20y%20Consignas/Proyecto%20Final/notebooks/mlruns/417185556600615154', creation_time=1762538423016, experiment_id='417185556600615154', last_update_time=1762538423016, lifecycle_stage='active', name='Proyectio_Properatio', tags={'mlflow.experimentKind': 'custom_model_development'}>

## 2. Transformación de variables numéricas con standarScaler

In [8]:

from sklearn.preprocessing import StandardScaler

numericas = ['rooms', 'bathrooms', 'surface_covered']
for col in numericas:
    scaler = StandardScaler()
    df[col] = scaler.fit_transform(df[[col]])


## 3 Separación de dataframes para distintos modelos

La variable del barrio se va a pasar por OHE para el modelo de LinearRegressor<br>
Para el de RandomForest se usará LabelEncoder en esa columna

In [9]:
from sklearn.preprocessing import LabelEncoder

### 3.2 LabelEncoder para RandomForest

In [10]:
# Versión para RandomForest
df_le = df.copy()

le = LabelEncoder()
df_le['neighborhood_encoded'] = le.fit_transform(df_le['neighborhood'])
df_le.drop(columns=['neighborhood'], inplace=True)

X_tree = df_le.drop(columns=['price_usd', 'price_per_m2'])
y_tree = df_le['price_usd']


### 3.3 OHE para LinearRegressor

In [11]:
# Versión para LinearRegressor
df_ohe = df.copy()

In [12]:
df_ohe.columns

Index(['neighborhood', 'rooms', 'bathrooms', 'surface_covered',
       'property_type', 'operation_type', 'price_usd', 'price_per_m2',
       'bathrooms_missing', 'rooms_missing'],
      dtype='object')

In [13]:
# Reemplazo los espacios por _ para mantener la consistencia en los nombres de columnas

df_ohe['neighborhood'] = df_ohe['neighborhood'].str.replace(' ', '_', regex=False)
df_ohe['neighborhood'] = df_ohe['neighborhood'].str.replace('/', 'o', regex=False)


# Paso todos los nombres de columnas a lowercase

df_ohe['neighborhood'] = df_ohe['neighborhood'].str.lower()

In [14]:
df_ohe['neighborhood'].unique()

array(['once', 'flores', 'palermo', 'tribunales', 'san_nicolás',
       'puerto_madero', 'centro_o_microcentro', 'almagro', 'barracas',
       'balvanera', 'chacarita', 'villa_crespo', 'san_cristobal',
       'villa_urquiza', 'retiro', 'recoleta', 'barrio_norte', 'congreso',
       'monserrat', 'san_telmo', 'colegiales', 'parque_patricios',
       'otros', 'caballito', 'catalinas', 'floresta', 'paternal',
       'belgrano', 'liniers', 'mataderos', 'nuñez', 'boca',
       'constitución', 'abasto', 'parque_chacabuco', 'boedo',
       'villa_devoto', 'villa_del_parque', 'saavedra'], dtype=object)

In [15]:
# Aplico OHE para los dataframes adaptados para LinearRegression
df_ohe = pd.get_dummies(df_ohe, columns=['neighborhood'], drop_first=True)
X_linear = df_ohe.drop(columns=['price_usd', 'price_per_m2'])
y_linear = df_ohe['price_usd']


In [16]:
#with open('models/columns_oficina_OHE.pkl', 'wb') as f:
#   pickle.dump(df_ohe.columns.tolist(), f)

## 4. Comparativa de modelos LinearRegressor y RandomForest, y métodos de transformación OHE y LabelEncoder

Se va a compara el rendimiento de 2 tipos de modelo distinto, 2 tipos de transformaciones, y las 4 combinaciones de datos posibles entre Alquiler, Venta, Local comercial y Oficina

Para medir el rendimiento según el porcentaje asignado a test, lo guardo como parámetro de mlFlow. <br>
Inicialmente separo un 70% de datos para el entrenamiento y un 30% para test

In [17]:
TEST_SIZE = 0.2
RANDOM_STATE = 24
mlflow.log_param("Tamaño de Test2", TEST_SIZE)
mlflow.log_param("Random state2", RANDOM_STATE)


24

In [18]:
df_ohe.sample(1)

,rooms,bathrooms,surface_covered,property_type,operation_type,price_usd,price_per_m2,bathrooms_missing,rooms_missing,neighborhood_almagro,...,neighborhood_retiro,neighborhood_saavedra,neighborhood_san_cristobal,neighborhood_san_nicolás,neighborhood_san_telmo,neighborhood_tribunales,neighborhood_villa_crespo,neighborhood_villa_del_parque,neighborhood_villa_devoto,neighborhood_villa_urquiza
14487,0.161744,-0.017358,1.013447,Oficina,Alquiler,21000.0,24.0,1,1,False,...,False,False,False,False,False,False,False,False,False,False


In [19]:
df_le.sample(1)

,rooms,bathrooms,surface_covered,property_type,operation_type,price_usd,price_per_m2,bathrooms_missing,rooms_missing,neighborhood_encoded
11061,-0.470024,0.687319,0.210465,Local comercial,Alquiler,8000.0,21.0,0,1,19


### 4.1 Entrenamiento y métricas iniciales

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, make_scorer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

combinations = [
    ('Oficina', 'Alquiler'),
    ('Oficina', 'Venta'),
    ('Local comercial', 'Alquiler'),
    ('Local comercial', 'Venta')
]
datasets = {'get_dummies': df_ohe, 'label_encoder': df_le}

results = []

for nombre_df, dataset in datasets.items():
    for tipo, operacion in combinations:
        # ----- LINEAR REGRESSION -----
        subset_lin = dataset[(dataset['property_type'] == tipo) & (dataset['operation_type'] == operacion)].copy()
        subset_lin = subset_lin.drop(columns=['property_type', 'operation_type'], axis=1)
        if not subset_lin.empty:
            X_lin = subset_lin.drop(columns=['price_usd', 'price_per_m2'])
            y_lin = subset_lin['price_usd']

            X_train, X_test, y_train, y_test = train_test_split(X_lin, y_lin, test_size=TEST_SIZE, random_state=RANDOM_STATE)

            model_lin = LinearRegression()
            model_lin.fit(X_train, y_train)
            y_pred = model_lin.predict(X_test)

            results.append({
                'Dataset': nombre_df,
                'Tipo': tipo,
                'Operación': operacion,
                'Modelo': 'LinearRegression',
                'R²': r2_score(y_test, y_pred),
                'RMSE': root_mean_squared_error(y_test, y_pred),
                'MAE': mean_absolute_error(y_test, y_pred)
            })

        # ----- RANDOM FOREST -----
        subset_tree = dataset[(dataset['property_type'] == tipo) & (dataset['operation_type'] == operacion)].copy()
        subset_tree = subset_tree.drop(columns=['property_type', 'operation_type'], axis=1)
        if not subset_tree.empty:
            X_tree = subset_tree.drop(columns=['price_usd', 'price_per_m2'])
            y_tree = subset_tree['price_usd']

            X_train, X_test, y_train, y_test = train_test_split(X_tree, y_tree, test_size=TEST_SIZE, random_state=RANDOM_STATE)

            model_rf = RandomForestRegressor(
                n_estimators=500,
                max_depth=None,
                max_features='sqrt',
                min_samples_leaf=2,
                min_samples_split=2,
                random_state=RANDOM_STATE
            )

            model_rf.fit(X_train, y_train)
            y_pred = model_rf.predict(X_test)

            results.append({
                'Dataset': nombre_df,
                'Tipo': tipo,
                'Operación': operacion,
                'Modelo': 'RandomForest',
                'R²': r2_score(y_test, y_pred),
                'RMSE': root_mean_squared_error(y_test, y_pred),
                'MAE': mean_absolute_error(y_test, y_pred)
            })

# ---- Tabla comparativa ----
results_df = pd.DataFrame(results)



In [21]:
results_df.sort_values(by='R²', ascending=False)

,Dataset,Tipo,Operación,Modelo,R²,RMSE,MAE
9,label_encoder,Oficina,Alquiler,RandomForest,0.831671,2618.001537,1043.555999
1,get_dummies,Oficina,Alquiler,RandomForest,0.826182,2660.339959,1117.588700
0,get_dummies,Oficina,Alquiler,LinearRegression,0.623348,3916.161131,1625.679010
8,label_encoder,Oficina,Alquiler,LinearRegression,0.593277,4069.487148,1630.500647
13,label_encoder,Local comercial,Alquiler,RandomForest,0.555348,2965.049775,795.002895
5,get_dummies,Local comercial,Alquiler,RandomForest,0.513829,3100.390467,809.563873
7,get_dummies,Local comercial,Venta,RandomForest,0.507718,425284.168572,170967.002787
15,label_encoder,Local comercial,Venta,RandomForest,0.454890,447521.973389,159105.718474
4,get_dummies,Local comercial,Alquiler,LinearRegression,0.451690,3292.568664,1012.154795
12,label_encoder,Local comercial,Alquiler,LinearRegression,0.430558,3355.415461,996.597903


Se observan mejores resultados usando el modelo RandomForest, mientras que respecto a la transformando la barrios no hay mucha diferencia entre GetDummies y LabelEncoder

#### 4.1.1 Validación cruzada

In [22]:
from sklearn.model_selection import cross_val_score, KFold


# Configuración general
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scorer_r2 = 'r2'
scorer_mae = 'neg_mean_absolute_error'

results = []

# Iteramos por tipo de propiedad y operación
for type, operation in combinations:
    # Dataset específico
    df_ohe_sub = df_ohe[
        (df_ohe['property_type'] == type) &
        (df_ohe['operation_type'] == operation)
    ]
    df_le_sub = df_le[
        (df_le['property_type'] == type) &
        (df_le['operation_type'] == operation)
    ]
    # Elimino columnas tipo Object
    df_ohe_sub = df_ohe_sub.drop(columns=['property_type', 'operation_type'], axis=1)
    df_le_sub = df_le_sub.drop(columns=['property_type', 'operation_type'], axis=1)

    # Separación X/y
    y_lin = df_ohe_sub['price_usd']
    X_lin = df_ohe_sub.drop(columns=['price_usd', 'price_per_m2'])
    y_tree = df_le_sub['price_usd']
    X_tree = df_le_sub.drop(columns=['price_usd', 'price_per_m2'])
    
    # Linear Regression
    lin_model = LinearRegression()
    scores_r2_lin = cross_val_score(lin_model, X_lin, y_lin, cv=kf, scoring=scorer_r2)
    scores_mae_lin = cross_val_score(lin_model, X_lin, y_lin, cv=kf, scoring=scorer_mae)

    results.append({
        "Tipo": type,
        "Operación": operation,
        "Modelo": "LinearRegression",
        "R² promedio": scores_r2_lin.mean(),
        "R² std": scores_r2_lin.std(),
        "MAE promedio": -scores_mae_lin.mean(),  # negado porque MAE se definió como "negativo"
        "MAE std": scores_mae_lin.std()
    })

    # Random Forest
    rf_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        max_features='sqrt',
        min_samples_split=2,
        min_samples_leaf=2,
        random_state=42
    )
    scores_r2_rf = cross_val_score(rf_model, X_tree, y_tree, cv=kf, scoring=scorer_r2)
    scores_mae_rf = cross_val_score(rf_model, X_tree, y_tree, cv=kf, scoring=scorer_mae)

    results.append({
        "Tipo": type,
        "Operación": operation,
        "Modelo": "RandomForest",
        "R² promedio": scores_r2_rf.mean(),
        "R² std": scores_r2_rf.std(),
        "MAE promedio": -scores_mae_rf.mean(),
        "MAE std": scores_mae_rf.std()
    })

# Tabla final
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["MAE promedio", "R² promedio"]).reset_index(drop=True)
results_df


,Tipo,Operación,Modelo,R² promedio,R² std,MAE promedio,MAE std
0,Local comercial,Alquiler,RandomForest,0.650792,0.049728,731.657985,26.230879
1,Local comercial,Alquiler,LinearRegression,0.585857,0.076806,966.357768,29.389439
2,Oficina,Alquiler,RandomForest,0.853484,0.020416,1089.451565,74.471242
3,Oficina,Alquiler,LinearRegression,0.782117,0.034221,1676.542813,62.603463
4,Oficina,Venta,RandomForest,0.592958,0.126380,115974.218275,17538.926300
5,Local comercial,Venta,RandomForest,0.426051,0.253387,168977.215321,10159.326952
6,Oficina,Venta,LinearRegression,0.539289,0.178116,191315.083317,12623.292780
7,Local comercial,Venta,LinearRegression,0.269876,0.275023,279778.737510,14718.083362


### 4.2 Un solo modelo de RandomForest podría estimar alquiler y venta? (2 en 1)

In [23]:
# Codifico operation_type (Alquiler = 0, Venta = 1) y la conservo como variable predictora,
# para que el modelo pueda distinguir entre alquileres y ventas
le = LabelEncoder()
df_le['operation_type'] = le.fit_transform(df_le['operation_type'])

In [24]:
tipo_propiedad = ['Oficina', 'Local comercial']
operaciones = {0: 'Alquiler', 1: 'Venta'}

def metricas_por_operacion(tp, X, y_log, y_pred_log, results):
    # R² global (escala log) y métricas por operación en la escala original (USD/m²)
    results.append({'Tipo': tp, 'Operación': 'Global', 'R²': r2_score(y_log, y_pred_log),
                    'MAE': np.nan, 'Mediana error %': np.nan})
    for code, name in operaciones.items():
        mask = (X['operation_type'] == code).values
        real, pred = np.expm1(y_log[mask]), np.expm1(y_pred_log[mask])
        results.append({'Tipo': tp, 'Operación': name,
                        'R²': r2_score(y_log[mask], y_pred_log[mask]),
                        'MAE': mean_absolute_error(real, pred),
                        'Mediana error %': np.median(np.abs(real - pred) / real * 100)})

results = []

for tp in tipo_propiedad:

    # ----- RANDOM FOREST -----
    subset_tree = df_le[df_le['property_type'] == tp].drop(columns=['property_type'])
    X_tree = subset_tree.drop(columns=['price_usd', 'price_per_m2'])  # conserva operation_type
    y_tree = np.log1p(subset_tree['price_per_m2'])

    X_train, X_test, y_train, y_test = train_test_split(X_tree, y_tree, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    model_rf_2in1 = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        max_features='sqrt',
        min_samples_leaf=2,
        min_samples_split=2,
        random_state=RANDOM_STATE
    )

    model_rf_2in1.fit(X_train, y_train)
    y_pred = model_rf_2in1.predict(X_test)

    metricas_por_operacion(tp, X_test, y_test.values, y_pred, results)

# ---- Tabla comparativa ----
results = pd.DataFrame(results)

In [25]:
results

,Tipo,Operación,R²,MAE,Mediana error %
0,Oficina,Global,0.972338,NaN,NaN
1,Oficina,Alquiler,0.568836,3.175272,26.693526
2,Oficina,Venta,0.434207,495.786027,14.232841
3,Local comercial,Global,0.957250,NaN,NaN
4,Local comercial,Alquiler,0.321324,2.923848,28.899613
5,Local comercial,Venta,0.347176,1008.827051,15.819403


**Lectura de resultados**

Al incluir `operation_type` como variable, el modelo conjunto alcanza un R² global de ≈ 0.97 (Oficinas) y ≈ 0.96 (Locales comerciales). Este valor no es comparable con los modelos separados: casi todo lo explica distinguir alquiler de venta (~7 USD/m² contra ~2.000 USD/m²). Para una comparación justa se evalúa cada operación por separado en la validación cruzada.

#### 4.2.1 Validación cruzada

In [26]:
from sklearn.model_selection import cross_val_predict

# Configuración general
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

# Iteramos por tipo de propiedad
for tp in tipo_propiedad:
    df_le_sub = df_le[df_le['property_type'] == tp].drop(columns=['property_type'])

    # Separación X/y
    y_tree = np.log1p(df_le_sub['price_per_m2'])
    X_tree = df_le_sub.drop(columns=['price_usd', 'price_per_m2'])  # conserva operation_type

    # Random Forest: predicción de cada fila con el fold en que quedó como validación
    rf_model_2in1 = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        max_features='sqrt',
        min_samples_split=2,
        min_samples_leaf=2,
        random_state=42
    )
    y_pred_cv = cross_val_predict(rf_model_2in1, X_tree, y_tree, cv=kf)

    metricas_por_operacion(tp, X_tree, y_tree.values, y_pred_cv, results)

# Tabla final
results_df = pd.DataFrame(results)
results_df

,Tipo,Operación,R²,MAE,Mediana error %
0,Oficina,Global,0.977281,NaN,NaN
1,Oficina,Alquiler,0.568537,3.107891,26.112075
2,Oficina,Venta,0.470095,525.798683,14.114217
3,Local comercial,Global,0.957408,NaN,NaN
4,Local comercial,Alquiler,0.334326,3.051384,31.258994
5,Local comercial,Venta,0.365507,1147.193589,17.560326


**Resultado de la comparación: modelo conjunto vs. modelos separados**

R² por operación, validación cruzada de 5 folds:

| Tipo | Operación | Modelo conjunto | Modelos separados (02_Models_comparisson, 4.1.1) |
|---|---|---|---|
| Oficina | Alquiler | 0.569 | 0.566 |
| Oficina | Venta | 0.470 | 0.499 |
| Local comercial | Alquiler | 0.334 | 0.343 |
| Local comercial | Venta | 0.366 | 0.373 |

- El modelo conjunto **no mejora** a los modelos separados: rinde igual o ligeramente peor en 3 de las 4 combinaciones.
- El R² global de ≈ 0.97 sólo refleja que el modelo distingue alquiler de venta, no que estime mejor el precio.
- Por simplicidad y por la diferencia de escala entre operaciones, se prefieren modelos separados.

## 5. Conclusiones

### Modelo elegido

Random Forest superó a Regresión Lineal en las 4 combinaciones de tipo de propiedad y operación. La combinación **Oficina en Alquiler** obtuvo el mejor desempeño en validación cruzada de 5 folds (R² = 0.85 ± 0.02, MAE ≈ 1.089 USD), coherente con el resultado en test (R² ≈ 0.83).

### Rendimiento

- El MAE no es comparable entre combinaciones, porque cada una tiene una escala de precio distinta; por eso la elección se apoya en el R².
- En ventas el rendimiento es bastante menor y con mayor variabilidad entre folds (R² 0.59 ± 0.13 en Oficinas y 0.43 ± 0.25 en Locales).

### Un modelo o dos

- Un modelo conjunto de alquiler y venta (incluyendo `operation_type` como variable) rinde igual o ligeramente peor que los modelos separados en cada operación.
- Además mezcla escalas de precio muy distintas, lo que hace poco informativa la métrica global.

### Decisión

- Se avanza con modelos separados, comenzando solamente con **Oficinas para Alquiler**.
- Queda la posibilidad de sumar a futuro modelos para predecir Ventas y para incluir Locales comerciales.
- Se probaron 2 test_size diferentes (0.2 y 0.3); 0.2 arrojó los mejores resultados.